<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" />

<h2>Charlynne Santos - Individual Assignment</h2>
<h3>Computational Data Analytics with Python</h3>
Hult International Business School<br>

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" />

<h3> Introduction </h3>
    
With the bike-sharing industry booming, reaching $2.8 billion globally in 2023, cities like Chicago face the challenge of optimizing their programs for efficiency and rider satisfaction. This analysis aims to predict daily bike rentals in a major U.S. city, specifically Chicago, considering key factors like time, weather, and holidays. Predicting demand allows for targeted resource allocation, such as deploying bikes to high-demand areas during peak hours or ensuring enough rentals are available during favorable weather conditions. This leads to reduced operational costs, increased rider satisfaction, and ultimately, a more sustainable and equitable transportation system that is convenient for all.

<b>Key highlights:</b><br>
<b>Data Analysis Insights:</b>
Weather's impact: The strong correlation between hour and rentals highlights the crucial role of timing. However, delving deeper,temperature and its interaction with hour is identified as the most significant predictors of bike rentals. This implies that pleasant weather conditions, especially during specific hours, drive higher demand.
Feature engineering success: By engineering features like Rush Hour, Heat Index, Wind Chill, and Temp Solar Interaction, the nuances of weather and its impact on bike rentals was captured. This demonstrates the value of moving beyond basic weather data to understand the complex interplay of factors influencing rider behavior.

<b>Modeling Results Insights:</b>
Six different models were developed and the Decision Tree model emerged as the most effective for predicting bike rentals, thanks to its ability to handle non-linear relationships and diverse data types. This aligns with the complexities of human decision-making influenced by various factors like weather and time.

<b>Overall Conclusion:</b>
This analysis sheds light on the critical role of time, weather, and specifically, the interaction between temperature and hour, in influencing bike rental demand. The superior performance of the Decision Tree model highlights the importance of considering these complex relationships for accurate predictions. By implementing these insights into bike-sharing programs, cities can optimize resource allocation, improve rider experience, and ultimately create a more sustainable and efficient transportation system.

<h3> Importation and Data Check </h3>

In [ ]:
## importing libraries ##

import numpy as np # Mathematical essentials
import pandas as pd # Data science essentials
import matplotlib.pyplot as plt # Visualization libraries
import seaborn as sns

# Scikit-learn libraries for modeling and preprocessing
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score


# setting pandas print options (optional)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)


In [ ]:
## importing data ##

# reading modeling data into Python
modeling_data = './datasets/train.xlsx'

# calling this df_trainx
df_full = pd.read_excel(io         = modeling_data,
                         sheet_name = 'data',
                         header     = 0,
                         index_col  = 'ID')


# checking data
df_full.head(n = 5)

<h3> Exploratory Data Analysis (EDA) and Data Preprocessing (DP) </h3>
<h4> Data Exploration </h4>

In [ ]:
# Understanding the Data Info
df_full.info()

==========================================================================================
<br>Missing values are present in Visibility, Dewpoint Temperature, and Solar Radiation. 
<br>Data Imputation will be done for these columns.
<br><br>DateHour, Holiday, FunctioningDay column data type is object, which might be critical parts of the analysis. Hence transformations will be needed for this.
==========================================================================================

In [ ]:
# Understanding the Data Statistics - numerical
df_full.describe()

In [ ]:
# Checking for updated distributions if the imputation shifted anything
imputed_columns = ['Visibility(miles)', 'DewPointTemperature(F)', 'SolarRadiation(MJ/m2)']

# Create a figure and a grid of subplots with a larger figsize
plt.figure(figsize=(10, 3))  # Adjust the size as needed

# Define a smaller font size for titles and labels
title_fontsize = 8
label_fontsize = 6

# Looping through the three columns and create a subplot for each one
for index, column in enumerate(imputed_columns, start=1): 
    plt.subplot(1, 3, index)  
    df_full[column].hist(bins= 30)
    plt.title(f'{column} Distribution', fontsize=title_fontsize)
    plt.xlabel(column, fontsize=label_fontsize)
    plt.ylabel('Frequency', fontsize=label_fontsize)
    plt.grid(False)
    plt.xticks(fontsize=label_fontsize)
    plt.yticks(fontsize=label_fontsize)

# Adjust the layout so that plots do not overlap
plt.tight_layout()

# Display the plots
plt.show()

==========================================================================================
<br> Looks like the distribution of visibility is skewed to a higher number. For this reason, median would be use for its imputation.
<br> DewpointTemp and solar seems a little bit more distributed. For this, we would use mean.
==========================================================================================

In [ ]:
# Understanding the Data Statistics - objects
df_full.describe(include = 'object')

==========================================================================================
<br>Columns Holiday and Functioning Day will can be transformed into binary (yes / no)
<br>DateHour will be transformed per month, hours, weekend
==========================================================================================

<h4> Data Imputation </h4>

In [ ]:
## Missing Value Check ##

# Checking for null values
# Get the information on missing values per column
missing_values_info = df_full.isnull().sum()

missing_values_info

In [ ]:
# Imputing data for Visibility, Temperature and Solar Radiation

# Imputing mean for Temperature
df_full['Visibility(miles)'].fillna(df_full['DewPointTemperature(F)'].median(), inplace=True)
    
# Imputing median for Visibility, and Solar Radiation
for column in ['SolarRadiation(MJ/m2)','DewPointTemperature(F)']:
    df_full[column].fillna(df_full[column].mean(), inplace=True)

# Verifying that there are no missing values left for the imputed columns
print(df_full[['Visibility(miles)', 'DewPointTemperature(F)', 'SolarRadiation(MJ/m2)']].isnull().sum())


In [ ]:
# Transforming DateHour, Holidays, and Functioning Day into numerical columns

# Applying transformation for DateHour as it is not in uniform formatting
def parse_dates(date_str):
    try:
        # Try parsing the '%Y-%m-%d %H:%M:%S.%f' format
        return pd.to_datetime(date_str, format='%Y-%m-%d %H:%M:%S.%f', errors='raise')
    except ValueError:
        # Fall back to the '%Y-%m-%d %H:%M:%S' format
        return pd.to_datetime(date_str, format='%Y-%m-%d %H:%M:%S', errors='raise')

# apply function
df_full['DateHour'] = df_full['DateHour'].apply(parse_dates)

# Extract 'Hour' and 'Month' from the datetime column and drop the original column
df_full['Hour'] = df_full['DateHour'].dt.hour # for hours
df_full['Month'] = df_full['DateHour'].dt.month # for month
df_full['weekday'] = df_full['DateHour'].dt.weekday # weekday
df_full['is_weekend'] = df_full['weekday'].apply(lambda x: 1 if x >= 5 else 0) # weekend

df_full.drop(['DateHour','weekday'], axis=1, inplace=True) # weekday is unncessecary since we have weekend

# Convert 'Holiday' and 'FunctioningDay' to binary variables
df_full['Holiday'] = df_full['Holiday'].map({'No': 0, 'Yes': 1})
df_full['FunctioningDay'] = df_full['FunctioningDay'].map({'No': 0, 'Yes': 1})

<h4> Current Data Analysis </h4>

In [ ]:
# Exploring timeframe trends for feature engineering
time_frames = ['Hour','Month', 'is_weekend', 'Holiday']

# Create a figure and a grid of subplots with a larger figsize
plt.figure(figsize=(10, 3))  # Adjust the size as needed

# Define a smaller font size for titles and labels
title_fontsize = 8
label_fontsize = 6

# Loop through the numerical columns and create a subplot for each one
for index, column in enumerate(time_frames, start=1):  # Start enumeration at 1 for subplot indexing
    plt.subplot(1, 4, index)  # Create a subplot in the right position
    df_full[column].hist(bins= 30)
    plt.title(f'{column} Distribution', fontsize=title_fontsize)
    plt.xlabel(column, fontsize=label_fontsize)
    plt.ylabel('Frequency', fontsize=label_fontsize)
    plt.grid(False)
    plt.xticks(fontsize=label_fontsize)
    plt.yticks(fontsize=label_fontsize)

# Adjust the layout so that plots do not overlap
plt.tight_layout()

# Display the plots
plt.show()


timeframes = ['Hour','Month', 'is_weekend', 'Holiday','RENTALS']
time_corr = df_full[timeframes].corr()
plt.figure(figsize=(8, 5))
sns.heatmap(time_corr, annot=True, fmt='.2f', cmap='coolwarm',linewidths=.5, cbar=True)
plt.title('Feature Correlation Matrix')
plt.show()


Distribution Analysis:
1. Hour Distribution: The hour distribution shows variance throughout different times of the day, which may correlate with typical rush hours.
2. Month Distribution: Although we only have a couple of months in our dataset, it is seen that there's variance too depending on the month. 
3. is_weekend: My hypothesis is that weekday distribution would slightly be similar but comparing it to weekend would show a difference. Based on the distribution, there is a shift in bikers during weekdays. Likely due school or work.
4. Holiday: Although the data we have on holiday is very low, it might still be worth it to put into considerations as there's more holidays in the future and can possibly still help with the model.

Correlation matrix:

1. Hour: As expected, there's a strong positive correlation between hour and rentals (0.42), indicating higher rentals during specific hours of the day. Further investigation could reveal peak rental times.
2. Month: The weak positive correlation (0.01) suggests that month might not have a significant standalone impact on rentals. However, it could be interesting to explore this further in conjunction with other factors like seasonality. However, seeing it as we do not have enough months in the dataset, factoring in seasonality might not help that much.
3. is_weekend: Weekends seem to have a slightly positive impact on rentals (0.07), but the correlation is weak. This could be due to people having more free time on weekends for leisure activities like cycling.
4. Holiday: unsurprisingly, there's a weak negative correlation between holidays and rentals (-0.03) as we have seen that our timeframe is low and the list of holidays on our dataset might be too low to really make out a pattern.

Overall, hour seems to be the most influential time frame for bike rentals, further analysis can be done to understand the nuances of hour-to-hour variations. Month, Holidays, and is_weekend unsurprisingly has low correlation and can be deprioritized later on during feature selection.

In [ ]:
## Analyzing Correlations of Weather Factors ##
# Select only numerical columns for distribution analysis
Weather_Factors = ['Temperature(F)', 'Humidity(%)', 'Wind speed (mph)', 
                'Visibility(miles)', 'DewPointTemperature(F)',  'Rainfall(in)', 
                'Snowfall(in)', 'SolarRadiation(MJ/m2)' ]

# Generate coorelation matrix heatmap
corr_matrix = df_full[Weather_Factors].corr()
plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',linewidths=.5, cbar=True)
plt.title('Feature Correlation Matrix')
plt.show()


Analysis on correlations, as it relates to RENTALS:

1. Temperature: A moderate positive correlation (0.72) with dew point suggests warmer temperatures (up to a point) can encourage bike rentals, as long as the humidity isn't too high. However, the negative correlation with rainfall (-0.22) indicates rainy days deter rentals.
2. Humidity: High humidity often goes hand-in-hand with cloudy skies and less sunlight, as shown by the strong negative correlation with solar radiation (-0.89). This, along with the moderate positive correlation with rainfall (0.38), explains why humidity can deter people from biking (0.21 negative correlation with rentals).
3. Wind Speed: While moderate wind can be refreshing, a strong positive correlation with solar radiation (0.61) suggests windier days might have clearer skies. However, the negative correlation with visibility (-0.05) implies potential safety concerns, and the overall impact on rentals is unclear with a weak positive correlation (0.21).
4. Visibility: This factor doesn't seem to have a major direct impact on bike rentals, as its correlations with other factors are weak (ranging from -0.37 to 0.02).
5. Dew Point: High dew point indicates muggy conditions, which might make people less inclined to rent bikes. This is supported by the moderate positive correlation with humidity (0.53) and the negative correlation with rentals (-0.34).
6. Rainfall: As expected, rainy days are a clear deterrent to bike rentals, reflected in the strong negative correlation (-0.53).
7. Snowfall: Snowfall significantly reduces bike rentals due to weather conditions, as shown by the strong negative correlation (-0.53).
8. Solar Radiation: More sunshine typically means better weather for biking, leading to potentially higher rentals. This is supported by the positive correlation with temperature (0.46) and the negative correlation with both humidity (-0.89) and rainfall (-0.40).

Based on this, it can be seen that pleasant weather with moderate temperatures (avoiding high dew point), low humidity, sunshine, and no rain or snow creates the best conditions for bike rentals. Having additional data related to might further improve the predictions.

<h3> Feature Engineering </h3>

Based on the results of the analysis I will focus my features on, hour and temperature interactions as the two has the highest correlations

<h4> Rush Hour Feature </h4>

As seen from the distribution before, there are certain peak hours that seems to entail higher rentals. To capture the increase in bike rental demand during typical commuting times, I will introduce a binary feature called Rush_Hour. This feature is set to 1 during peak traffic hours, I've defined as 6-9 AM and 4-6 PM, and 0 otherwise. The rationale behind this is to pinpoint the parts of the day when people are most likely to rent bikes for commuting to and from work or school. This timeframe captures the morning rush to work and the evening return and can factor in a lot during rentals.

Importance: why this feature is added is because the hour value from 0 to 24 alone will not capture specific time frames of rush hours where it is expected to have a high demand. Hence this feature strengten the rush hour predictions.

In [ ]:
# Create rush hour feature using Hour
df_full['Rush_Hour'] = df_full['Hour'].apply(lambda x: 1 if (6 <= x <= 9) or (16 <= x <= 18) else 0)

<h4> Comfort-Related Features </h4>

From the correlation matrix above and based on real life experience, I hypothesize that weather conditions significantly impact bike rental patterns, as extreme conditions could deter people from outdoor activities. 

To this end, I've created several features that combine temperature with other weather-related variables. Temperature is the key factor here as we saw its high correlation between rentals, and in reality it is probably the first basis of people whenever they go out for a bike. At least for me it is.

Comfort related features are:

1. Heat Index: This feature is a simplified proxy for the "feels like" temperature, accounting for both heat and humidity. It is important to note that this is not the actual heat index calculation, which is more complex, but it should still serve as a good indicator of discomfort during hot and humid conditions.

2. Wind Chill: The wind chill feature is intended to represent the perceived cold due to wind speed. Similar to the heat index, this is a simplified calculation and is expected to highlight times when the wind makes the temperature feel cooler than it actually is, which may influence the decision to rent a bike.

3. Temp Solar Interaction: We also consider the interaction between temperature and solar radiation. This feature aims to capture the increased likelihood of bike rentals on days that are both warm and sunny, as opposed to warm but overcast days. 

Importance: By incorporating comfort-related features such as Heat Index, Wind Chill, and Temp Solar Interaction into our dataset, we gain granular insights into the weather conditions, providing a better view that goes beyond mere temperature readings. These features consider the interplay of various weather elements and how they collectively impact human comfort, which is a significant determinant in bike rental decisions. Understanding and quantifying comfort levels are essential, as people are naturally more inclined to rent bikes when the conditions are pleasant. Moreover, these features allow for the capture of complex interactions between weather variables, offering a multi-dimensional perspective that enhances model accuracy. Such an approach acknowledges the behavioral influence on rental patterns and leverages the synergy of multiple weather factors, leading to a more sophisticated and precise prediction model.

In [ ]:
# Temperature-related comfort features
df_full['Heat_Index'] = df_full['Temperature(F)'] * df_full['Humidity(%)']  # Simplified example
df_full['Wind_Chill'] = df_full['Temperature(F)'] * df_full['Wind speed (mph)']  # Simplified example
df_full['Temp_Solar'] = df_full['Temperature(F)'] * df_full['SolarRadiation(MJ/m2)']  # Temperature and solar radiation interaction

<h4> Interaction of Temperature and Hour </h4>

Lastly, I have created an Interaction_temp_hour feature to explore how the combination of temperature and the time of day affects bike rentals. We know that weather plays a significant role in the decision to rent a bike, and this role may change throughout the day.

Importance: both hour and temperature has been proven as the most siginifcant based on the correlation matrix. By introducing this interaction term, we allow the model to learn different patterns at different times of the day, which could be crucial for forecasting demand accurately. Since we have only incorporated rush hours by checking the relationship of temperature and hour we might find a pattern where people go for bikes across the day due to better weather (temperature) conditions. 

In [ ]:
# Temperature and hour interaction  features
df_full['Interaction_temp_hour'] = df_full['Temperature(F)'] * df_full['Hour']

In [ ]:
## Features vs Rentals ##
# Select only numerical columns for distribution analysis
features = ['RENTALS',
    'Rush_Hour',
    'Heat_Index',
    'Wind_Chill',
    'Temp_Solar',
    'Interaction_temp_hour'
]

# Generate coorelation matrix heatmap
corr_matrix = df_full[features].corr()
plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',linewidths=.5, cbar=True)
plt.title('Feature Correlation Matrix')
plt.show()


Analysis:

1. Rush_Hour and RENTALS (0.273959): There is a positive correlation, which suggests that bike rentals tend to increase during rush hours. This makes sense as more people may rent bikes to commute to and from work.
2. Heat_Index and RENTALS (-0.168299): This negative correlation indicates that higher heat index values, which represent hotter perceived temperatures, might lead to a decrease in bike rentals. It could be that extreme heat is a deterrent for biking.
3. Wind_Chill and RENTALS (0.266276): There is a positive correlation, albeit a little unexpected, because one might think colder wind chill values would deter bike rentals. However, this positive relationship may indicate that in the range of temperatures experienced, wind chill does not have a strongly negative effect, or it could be related to seasons where wind chill is higher along with other favorable conditions for biking.
4. Temp_Solar and RENTALS (0.223431): There is a positive correlation, which can be interpreted to mean that when it's sunnier, and potentially the weather is more pleasant, more people are likely to rent bikes.
5. Interaction_temp_hour and RENTALS (0.499735): This is the strongest correlation observed, suggesting that the interaction between temperature and hour has a significant impact on bike rentals. This makes sense as certain hours of the day with optimal temperatures could be ideal for biking, leading to more rentals.


From these correlations, the Interaction_temp_hour feature stands out as the most strongly correlated with bike rentals, indicating that it might be a very important predictor during the modeling. On the otherhand, the lower correlation such as heat index might not make much improvement in the model. 

This will be taken into account as I proceed with modeling

<h3> Candidate Model Development</h3>

<h4>Train-Test Split</h4>

For feature selection, I started out as using all the features and removing one by one by correlation to decide which would be retained.

In [ ]:
# Prepare the training data for modeling

#x_features = df_full.drop(columns=['RENTALS'])
y_variable = df_full['RENTALS']

core_features = [
                'Temperature(F)', 
                'Humidity(%)', 
                'Wind speed (mph)', 
                'Visibility(miles)', 
                'DewPointTemperature(F)', 
                'Rainfall(in)', 
                'Snowfall(in)', 
                'SolarRadiation(MJ/m2)', 
                'Holiday', 
                'FunctioningDay',
                'Hour', 
                'Month',
                'is_weekend'
]

time_frame_features = [
                'Rush_Hour'
]

comfort_features = [
    'Heat_Index',
    'Wind_Chill',
    'Temp_Solar',
]

interaction_features = [
    'Interaction_temp_hour',
]

x_features = df_full[core_features + time_frame_features + comfort_features + interaction_features]



<h4>Candidate Modeling Results</h4>

Using for loop to summarize results across all models and make the comparison simpler

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_features, y_variable, test_size=0.3, random_state=8)

# Candidate Model Development
models = {
    'Linear Regression': LinearRegression(),
    'Lasso': Lasso(alpha=0.1),
    'Ridge': Ridge(alpha=0.1),
    'Elastic Net': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'Decision Tree': DecisionTreeRegressor(max_depth = 10, max_features = None, min_samples_leaf = 2, min_samples_split = 30)
}

# Train and evaluate each model
for name, model in models.items():
    model.fit(x_train, y_train)
    
    # Training scores
    y_train_pred = model.predict(x_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    
    # Testing scores
    y_test_pred = model.predict(x_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Gap between training and testing scores
    rmse_gap = abs(train_rmse - test_rmse)
    r2_gap = abs(train_r2 - test_r2)
    
    # Print the results
    print(f"""{name}:
    Train RMSE: {train_rmse:.4f}\tTest RMSE: {test_rmse:.4f}\tRMSE Gap: {rmse_gap:.4f}
    Train R2 Score: {train_r2:.4f}\tTest R2 Score: {test_r2:.4f}\tR2 Score Gap: {r2_gap:.4f}""")
    print("-" * 80)


<b>Candidate Modeling Results:</b>

<b>Linear Regression:</b>
The model has a moderate fit with an R2 score of 0.6883 on the training set and 0.6155 on the test set. The gap between the train and test R2 scores suggests some overfitting.
The RMSE values are relatively high, indicating that the model's predictions have some error.

<b>Lasso:</b>
Lasso performs similarly to Linear Regression, with a slight improvement in the test RMSE and R2 score. The regularization in Lasso helps to reduce overfitting slightly.

<b>Ridge:</b>
Ridge shows a further improvement in test performance compared to Lasso, with the smallest RMSE gap and R2 score gap among the linear models. This indicates that Ridge regularization is effective in reducing overfitting while maintaining predictive performance.

<b>Elastic Net:</b>
Elastic Net has a higher RMSE and lower R2 score compared to the other linear models, suggesting it may not be the best fit for this data. However, it has the smallest gap between train and test RMSE, indicating less overfitting.

<b>K-Nearest Neighbors (KNN):</b>
KNN has the highest RMSE values and the lowest R2 scores, indicating it is the least effective model for this dataset. The large gap between train and test scores suggests significant overfitting. This might be because I did not use any standardization at all with the data.

<b>Decision Tree:</b>
The Decision Tree model has the lowest RMSE values and the highest R2 scores, indicating it is the best performing model among those tested. It also shows a relatively small gap between train and test scores, suggesting it generalizes well to unseen data.

Overall, the Decision Tree model seems to be the most effective for predicting bike rentals in this dataset, with the best balance between performance and generalization. The linear models (Ridge, Lasso, and Linear Regression) show moderate performance, with Ridge being the best among them. Elastic Net and KNN appear to be less suitable for this particular problem.

With these the top 3 model to be used for hyperparameter tuning would be:
1. Decision Tree
2. Ridge
3. Linear Regression

<h4> Hyperparameter Tuning </h4>

Using for loop to summarize results across all models and make the comparison simpler

In [ ]:
# Hyperparameter Tuning ##

## Top 1 Model ##
# Decision Tree Regressor
dt_params = {'max_depth': [None, 10, 20, 30, 40, 50, 60],
    'min_samples_split': [2, 5, 10, 15, 20, 30, 40],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 'log2', None] }
dt_grid = GridSearchCV(DecisionTreeRegressor(), dt_params, cv=5, scoring='neg_mean_squared_error')
dt_grid.fit(x_train, y_train)
dt_best = dt_grid.best_estimator_


## Top 2 Model ##
# Ridge Regression
ridge_params = {'alpha': np.logspace(-2, 4, 70)}
ridge_grid = GridSearchCV(Ridge(), ridge_params, cv=5, scoring='neg_mean_squared_error')
ridge_grid.fit(x_train, y_train)
ridge_best = ridge_grid.best_estimator_

## Top 3 Model ##
# Third highest model is linear regression

# Evaluate the tuned models
tuned_models = {
    'Decision Tree (Tuned)': dt_best,
    'Linear Regression': models['Linear Regression'],
    'Ridge (Tuned)': ridge_best
}

tuned_results = []

# generate results in a loop form
for name, model in tuned_models.items():
    train_pred = model.predict(x_train) 
    test_pred = model.predict(x_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred)) # generating RSME to evaluate results better
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    gap = train_r2 - test_r2 # computing for score gap to ensure final model fall below 0.05 threshold
    tuned_results.append({'Model': name, 'Train RMSE': train_rmse, 'Test RMSE': test_rmse, 'Train R2': train_r2, 'Test R2': test_r2, 'Score Gap': gap})

# Display the results after tuning, sorted by Test RMSE
print("Parameter Tuned Model results:")
tuned_results_df = pd.DataFrame(tuned_results)
tuned_results_df.sort_values(by='Test RMSE', ascending=True, inplace=True)
print(tuned_results_df)


<b>Hyper Parameter Tuning Results:</b>

<b>Decision Tree (Tuned):</b>
The tuned Decision Tree model shows a slight increase in both train and test RMSE compared to the untuned version. However, the increase is minimal, and the model still maintains a high R2 score.
The gap between the train and test R2 scores has increased slightly, indicating a small increase in overfitting. However, the model still performs well overall and it is the only one that has the smallest score gap between training and testing.

<b>Ridge (Tuned):</b>
The tuned Ridge model shows a slight increase in both train and test RMSE compared to the untuned version. However, the increase in test RMSE is smaller than the increase in train RMSE, indicating an improvement in generalization.
The R2 score on the test set has improved, and the gap between the train and test R2 scores has decreased, suggesting that the tuning has effectively reduced overfitting.

<b>Linear Regression:</b>
The Linear Regression model remains unchanged, serving as a baseline for comparison.
Overall, the tuning has led to improvements in the generalization of the Ridge model, as evidenced by the reduced gap between train and test scores. The Decision Tree model still performs well, despite a slight increase in overfitting. It remains the best performing model among those tested, with the highest R2 scores and lowest RMSE values.

<b>Final Model: Based on this, it is decided that from the features I have created and the models I have used, the best model to use for prediction would be Decision Tree Regressor. As it works by leveraging a series of simple questions and branching logic, Decision Trees effectively capture non-linear relationships and handle various data types, making them well-suited for bike rental prediction (Kawsar, 2021)<b/>.



<h3> Preparing for Kaggle Submission </h3>

<h4> Loading and Processing Data for Prediction </h4>

In [ ]:
# Load and preprocess the test dataset
kaggle_data = pd.read_excel(io         = './datasets/test.xlsx',
                        sheet_name = 'data',
                        header     = 0,
                        index_col  = 'ID')


# Impute missing values
kaggle_data['Visibility(miles)'].fillna(kaggle_data['Visibility(miles)'].median(), inplace=True)
for column in ['DewPointTemperature(F)', 'SolarRadiation(MJ/m2)']:
    kaggle_data[column].fillna(kaggle_data[column].mean(), inplace=True)

# Convert DateHour and Holiday
kaggle_data['DateHour'] = kaggle_data['DateHour'].apply(parse_dates)
kaggle_data['Hour'] = kaggle_data['DateHour'].dt.hour
kaggle_data['Month'] = kaggle_data['DateHour'].dt.month
kaggle_data['weekday'] = kaggle_data['DateHour'].dt.weekday
kaggle_data['is_weekend'] = kaggle_data['weekday'].apply(lambda x: 1 if x >= 5 else 0)
kaggle_data.drop(['DateHour','weekday'], axis=1, inplace=True)

# Doing the same for Holidays and functioning day
kaggle_data['Holiday'] = kaggle_data['Holiday'].map({'No': 0, 'Yes': 1})
kaggle_data['FunctioningDay'] = kaggle_data['FunctioningDay'].map({'No': 0, 'Yes': 1})

# Feature Engineering
kaggle_data['Rush_Hour'] = kaggle_data['Hour'].apply(lambda x: 1 if (6 <= x <= 9) or (16 <= x <= 18) else 0)
kaggle_data['FeelsLikeTemperature(F)'] = kaggle_data['Temperature(F)'] - 0.55 * (1 - kaggle_data['Humidity(%)'] / 100) * (kaggle_data['Temperature(F)'] - kaggle_data['DewPointTemperature(F)'])
kaggle_data['Heat_Index'] = kaggle_data['Temperature(F)'] * kaggle_data['Humidity(%)']
kaggle_data['Wind_Chill'] = kaggle_data['Temperature(F)'] * kaggle_data['Wind speed (mph)']
kaggle_data['Temp_Solar'] = kaggle_data['Temperature(F)'] * kaggle_data['SolarRadiation(MJ/m2)']
kaggle_data['Interaction_temp_hour'] = kaggle_data['Temperature(F)'] * kaggle_data['Hour']

# Prepare x_data_kaggle
x_data_kaggle = kaggle_data[core_features + time_frame_features + comfort_features + interaction_features]


<h4> Predicting for Kaggle Submission </h4>

In [ ]:
# Train the Decision Tree model on the entire training dataset
dt_best.fit(x_features, y_variable)

# Make predictions using the best Decision Tree model
predictions = dt_best.predict(x_data_kaggle)

## Kaggle Submission File ##

# organizing predictions
model_predictions = {"RENTALS" : predictions}


# converting predictions into df
model_pred_df = pd.DataFrame(data  = model_predictions,
                             index = kaggle_data.index)

# name your model
model_pred_df.to_csv(path_or_buf = "./model_output/FINAL_SUBMITTED.csv",
                     index       = True,
                     index_label = 'ID')

<h3> References </h3>
    
Kawsar, S. (2021, June 1). Machine Learning Quiz 05: Decision Tree (Part 1). Medium. https://kawsar34.medium.com/machine-learning-quiz-05-decision-tree-part-1-3ea71fa312e5
    
OpenAI. (2022). ChatGPT (Dec 20 version). Date Accessed: February 22, 2024. [Large language model]. https://chat.openai.com/

<h3> Appendix </h3>
<br><b>Chatgpt prompts</b>
<br>A. DateHour error code check
<br>B. Some of the codes in hyperparameter tuning:
<br>   1. How can I further improve my DecisionTree and RIDGE with hyper parameter tuning
<br>   2. How do I code searchgrid for both of it and what would be my parameters
<br>   3. Could you help me print the results that contains  Model, Train RMSE, Test RMSE, Train R2, Test R2, Score Gap